In [ ]:
# Linear Regression without GridSearch But with Cross Validation
import pandas as pd
import numpy as np
from time import time

from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge, Lasso, SGDRegressor
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.model_selection import RepeatedKFold
from sklearn.model_selection import GridSearchCV
from sklearn import metrics
from sklearn.metrics import explained_variance_score,mean_absolute_error,r2_score


bostondata = pd.read_csv('boston.csv')
X = bostondata.drop('medv',axis=1)
y = bostondata['medv']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state= 101)   # 0.3 is standard test size, pick what you need to

print (X_train.shape, y_train.shape)
print (X_test.shape, y_test.shape)    #It's good practice to check

lm = LinearRegression()

#Next we do cross validation, which splits apart our training data and fits the model on different samples and
# gives scores for each sample to get the best fit model before we test it on the testing data.

scores = cross_val_score(lm, X_train, y_train, cv = 5)    #cv is the number of folds, scores will give an array of scores

print ('Scores: ', scores, '\nMean Score: ', np.mean(scores), '\nSTD of Scores: ', np.std(scores))

#To get predictions (y_hat) and check them all in one using cross validation

predictions = cross_val_predict(lm, X_test, y_test, cv = 5)     #y_test is needed here in predictions to get scores for each fold of cv

accuracy = metrics.r2_score(y_test, predictions)  #this says the accuracy of the predictions from the best cv fold


#If this is good, continue to fit the model on the data


lm.fit(X_train, y_train)

y_hat = lm.predict(X_test)      #this gives me my predictions

print('\n\nmodel performance: ', lm.score(X_test, y_test))    #this tells me my model performance

(354, 14) (354,)
(152, 14) (152,)
Scores:  [0.6573394  0.70650294 0.74205027 0.78461421 0.61969732] 
Mean Score:  0.7020408296660389 
STD of Scores:  0.05868168362062682


model performance:  0.7124037966603735


In [ ]:

# Linear Regression (The SGD Regressor) with GridSearchCV and RepeatedKFold CrossValidation

X1 = bostondata.drop('medv',axis=1)
y1 = bostondata['medv']

# define model
model = SGDRegressor()

cv = RepeatedKFold(n_splits=10, n_repeats=3) #RepeatedKFold means repeating KFold with different random state each time


## Ridge Regression
# For more information, you can visit this documentation.

# define parameters
param = {
    'penalty':['l1', 'l2', 'elasticnet'],
    'alpha': [1e-4, 1e-3, 1e-2, 1e-1],
    'fit_intercept':[True, False],
    'eta0':[0.1, 0.01, 0.001]
}


# define search
search = GridSearchCV(model, param, scoring='neg_root_mean_squared_error', n_jobs=-1, cv=cv)
# execute search
result = search.fit(X1, y1)

# summarize result

print('Best Score: %s' % result.best_score_)
print('Best Hyperparameters: %s' % result.best_params_)

Best Score: -12322382955193.436
Best Hyperparameters: {'alpha': 0.1, 'eta0': 0.001, 'fit_intercept': False, 'penalty': 'elasticnet'}


In [ ]:
bostondata['medv']

0      24.0
1      21.6
2      34.7
3      33.4
4      36.2
       ... 
501    22.4
502    20.6
503    23.9
504    22.0
505    11.9
Name: medv, Length: 506, dtype: float64

In [ ]:
# Apply Scaling
X2 = bostondata.drop('medv',axis=1)
y2 = bostondata['medv']

#Perform Scaling
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_sc = sc.fit_transform(X2)

#Splitting the data into train and test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_sc, y2, test_size=0.2, random_state=42)



In [ ]:
regressors = [
    LinearRegression(),
    Lasso(),
    Ridge(),
    SGDRegressor(alpha=0.1, eta0= 0.001, fit_intercept= False, penalty = 'elasticnet'),
    SGDRegressor()
]

for model in regressors[:5]:
    start = time()
    model.fit(X_train, y_train)
    train_time = time() - start
    start = time()
    y_pred = model.predict(X_test)
    predict_time = time()-start
    print(model)
    print("\tTraining time: %0.3fs" % train_time)
    print("\tPrediction time: %0.3fs" % predict_time)
    print("\tExplained variance:", explained_variance_score(y_test, y_pred))
    print("\tMean absolute error:", mean_absolute_error(y_test, y_pred))
    print("\tR2 score:", r2_score(y_test, y_pred))
    print()

LinearRegression()
	Training time: 0.006s
	Prediction time: 0.000s
	Explained variance: 0.6667009534619168
	Mean absolute error: 3.2007547573408206
	R2 score: 0.6659408703343037

Lasso()
	Training time: 0.003s
	Prediction time: 0.000s
	Explained variance: 0.6248316409950724
	Mean absolute error: 3.46484952823374
	R2 score: 0.6242880160354785

Ridge()
	Training time: 0.006s
	Prediction time: 0.000s
	Explained variance: 0.6665968598157075
	Mean absolute error: 3.1971747130988706
	R2 score: 0.665822902814839

SGDRegressor(alpha=0.1, eta0=0.001, fit_intercept=False, penalty='elasticnet')
	Training time: 0.008s
	Prediction time: 0.000s
	Explained variance: 0.6221068913977592
	Mean absolute error: 23.30665629786999
	R2 score: -6.785123506078036

SGDRegressor()
	Training time: 0.002s
	Prediction time: 0.000s
	Explained variance: 0.653063820999309
	Mean absolute error: 3.242451629418228
	R2 score: 0.6517764998341204

